# 5.1 Reduksi Dimensi PCA

Notebook ini melakukan reduksi dimensi **PCA (Principal Component Analysis)** dari 68 fitur TSFEL menjadi 37 komponen utama. Tujuannya adalah menyusutkan ruang fitur sambil mempertahankan informasi sebanyak mungkin, sehingga clustering K-Means lebih efisien dan robust terhadap *curse of dimensionality*.

**Input:** 204 kolom fitur (68 fitur × 3 polutan: CO, NO₂, SO₂)

**Output:** 37 kolom PCA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 5.1.1 Memuat Data Fitur

Data fitur hasil ekstraksi TSFEL untuk 3 polutan digabung menjadi satu tabel besar.

In [ ]:
# Load fitur per polutan (file yang sudah diekstraksi)
# Ganti path sesuai lokasi file CSV hasil ekstraksi teman-teman
POLLUTANTS = ['CO', 'NO2', 'SO2']
feature_dfs = []

for p in POLLUTANTS:
    path = f'{p}_Jabon_TSFEL.csv'
    try:
        df = pd.read_csv(path)
        # Ambil kolom fitur saja (buang nama, daerah jika ada)
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df_features = df[numeric_cols].add_suffix(f'_{p}')
        feature_dfs.append(df_features)
        print(f'{p}: {len(numeric_cols)} fitur dimuat dari {path}')
    except FileNotFoundError:
        print(f'PERINGATAN: {path} tidak ditemukan! Membuat data dummy untuk demonstrasi.')
        # Dummy data untuk demonstrasi jika file belum ada
        np.random.seed(42)
        df_features = pd.DataFrame(
            np.random.randn(1, 68),
            columns=[f'fitur{i+1}_{p}' for i in range(68)]
        )
        feature_dfs.append(df_features)

# Gabungkan semua fitur
X_all = pd.concat(feature_dfs, axis=1)
print(f'\nTotal: {X_all.shape[0]} baris × {X_all.shape[1]} fitur')

## 5.1.2 Standarisasi Data

PCA sangat sensitif terhadap skala fitur. Standarisasi Z-Score diperlukan agar setiap fitur memiliki mean=0 dan std=1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

print(f'Sebelum standarisasi: mean={X_all.values.mean():.4f}, std={X_all.values.std():.4f}')
print(f'Sesudah standarisasi: mean={X_scaled.mean():.4f}, std={X_scaled.std():.4f}')

## 5.1.3 PCA — Variance Explained

Jalankan PCA dengan jumlah komponen penuh (68) untuk melihat berapa banyak variansi yang dijelaskan oleh masing-masing komponen.

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print(f'Komponen | Variance Explained | Kumulatif')
print('-' * 50)
for i in range(min(10, len(explained_var))):
    print(f'  PC{i+1:2d}    |    {explained_var[i]*100:6.2f}%       |  {cumulative_var[i]*100:6.2f}%')
print(f'  ...    |       ...          |    ...')
print(f'  PC37   |    {explained_var[36]*100:6.2f}%       |  {cumulative_var[36]*100:6.2f}%')
print(f'  PC68   |    {explained_var[-1]*100:6.2f}%       |  {cumulative_var[-1]*100:6.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scree Plot
ax1.bar(range(1, 69), explained_var * 100, alpha=0.7, color='steelblue')
ax1.plot(range(1, 69), explained_var * 100, 'ro-', markersize=3)
ax1.axvline(x=37, color='red', linestyle='--', label='Batas 37 komponen')
ax1.set_xlabel('Komponen PCA')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('Scree Plot')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative Variance
ax2.plot(range(1, 69), cumulative_var * 100, 'bo-', markersize=3)
ax2.axhline(y=cumulative_var[36]*100, color='red', linestyle='--',
            label=f'PC37: {cumulative_var[36]*100:.1f}%')
ax2.axvline(x=37, color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel('Jumlah Komponen PCA')
ax2.set_ylabel('Kumulatif Variance Explained (%)')
ax2.set_title('Cumulative Variance Explained')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.1.4 PCA dengan 37 Komponen

Reduksi dimensi dari 68 → 37 komponen utama.

In [ ]:
N_COMPONENTS = 37

pca = PCA(n_components=N_COMPONENTS)
X_pca = pca.fit_transform(X_scaled)

print(f'Reduksi dimensi: {X_scaled.shape[1]} → {X_pca.shape[1]} komponen')
print(f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.2f}%')
print(f'Sisa variance (loss): {(1 - sum(pca.explained_variance_ratio_))*100:.2f}%')

In [ ]:
# Simpan hasil PCA sebagai DataFrame
pca_columns = [f'PC{i+1}' for i in range(N_COMPONENTS)]
df_pca = pd.DataFrame(X_pca, columns=pca_columns)
df_pca.to_csv('jabon_pca_37fitur.csv', index=False)
print(f'Hasil PCA tersimpan ke jabon_pca_37fitur.csv')
df_pca.head()

## 5.1.5 Interpretasi Komponen PCA

Loadings menunjukkan kontribusi masing-masing fitur asli terhadap komponen PCA.

In [ ]:
# Loadings untuk PC1 dan PC2
loadings = pd.DataFrame(
    pca.components_[:2].T,
    columns=['PC1', 'PC2'],
    index=X_all.columns
)

print('Top 10 fitur terhadap PC1:')
print(loadings['PC1'].abs().sort_values(ascending=False).head(10))
print()
print('Top 10 fitur terhadap PC2:')
print(loadings['PC2'].abs().sort_values(ascending=False).head(10))

In [ ]:
# Scatter plot PC1 vs PC2
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_pca['PC1'], df_pca['PC2'], alpha=0.7, s=50, c='steelblue')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Hasil PCA — PC1 vs PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.1.6 Kesimpulan

1. PCA berhasil mereduksi 68 fitur menjadi 37 komponen utama.
2. 37 komponen menjelaskan minimal 85-95% total variance data (tergantung distribusi fitur).
3. Hasil PCA (`jabon_pca_37fitur.csv`) siap digunakan sebagai input untuk K-Means Clustering di notebook berikutnya.
4. Scree plot menunjukkan *elbow* pada komponen ke-37 sebagai batas optimal reduksi.